In [0]:
#!pip install statsmodels
#!pip install linearmodels
#!pip install lseg.data
!pip install pandas_datareader

In [0]:
# Fetch Apple stock prices for the last year using ld API
import lseg.data as ld
from datetime import datetime, timedelta

# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
import pandas as pd
import numpy as np
import math
from statsmodels.api import OLS, add_constant
import pandas_datareader.data as web
from linearmodels.asset_pricing import LinearFactorModel

import matplotlib.pyplot as plt
import seaborn as sns



In [0]:
sns.set_style('whitegrid')

In [0]:
ff_factor = 'F-F_Research_Data_5_Factors_2x3'
ff_factor_data = web.DataReader(ff_factor, 'famafrench', start='2013', end='2026-07')[0]
ff_factor_data.info()

In [0]:
ff_factor_data

In [0]:
ff_factor_data.describe()

In [0]:
dji = ld.get_data('0#.DJI', fields=['TR.CommonName', 'TR.PriceClose',
                                      'TR.Volume', 'TR.TotalReturnYTD'])

In [0]:
dji

In [0]:
const = dji['Instrument'].tolist()
const_limp = [i for i in const if i != '']
const_limp

In [0]:
from datetime import datetime, timedelta
end = '2026-08-05'
start = '2012-12-01'
print(start, end)

In [0]:
df = ld.get_data(const_limp[0], ['TR.TotalReturn.date', "TR.ClosePrice", 'TR.TotalReturn'],
{'SDate': start, 'EDate': end, 'Frq': 'M' })
df

In [0]:
df = ld.get_data(const_limp[0], ['TR.TotalReturn.date', "TR.ClosePrice", 'TR.TotalReturn'],
{'SDate': start, 'EDate': end, 'Frq': 'M' })
df[const_limp[0]] = np.log((df['Total Return']/100)+1)
monthly_returns = df.drop(['Close Price', 'Total Return', 'Instrument'], axis=1).iloc[1:]
monthly_returns.set_index('Date', inplace = True)
monthly_returns

In [0]:
meses = monthly_returns.count().iloc[0]
meses

In [0]:
for i in const_limp[1:]:
    w = ld.get_data(i, ['TR.TotalReturn.date', "TR.ClosePrice", 'TR.TotalReturn'],
{'SDate': start, 'EDate': end, 'Frq': 'M' })
    w = w[w['Total Return'] != 'NaN']
    w[i] = np.log((w['Total Return'].astype(float)/100)+1)
    w1 = w.drop(['Close Price', 'Total Return', 'Instrument'], axis=1).iloc[1:]
    p = round(w1.count().iloc[0])
    print(i, p)
    if p == meses:    
        monthly_returns = monthly_returns.join(w1.set_index('Date'))

In [0]:
monthly_returns

In [0]:
df2=monthly_returns.reset_index()
df2['Date'] = pd.to_datetime(df2['Date']).dt.to_period('M')
returns = df2.set_index('Date')
returns

In [0]:
excess_returns = returns.sub(ff_factor_data.RF, axis=0)
excess_returns

In [0]:
excess_returns = excess_returns.clip(lower=np.percentile(excess_returns, 1),
                                     upper=np.percentile(excess_returns, 99))

# excess_returns

In [0]:
const = dji['Instrument'].tolist()
const_limp = [i for i in const if i != '']
const_limp

In [0]:
betas = []
for i in excess_returns.columns:
    step1 = OLS(endog=excess_returns.loc[:,i].astype(float), 
                exog=add_constant(ff_factor_data)).fit()
    betas.append(step1.params.drop('const'))
betas

## Y es el rendimiento de comapañía i  = B0 Mft-RF + B1 SMB + B2 HML + B3 RMW + B4 CMA + B5 RF + e


In [0]:
betas = pd.DataFrame(betas, 
                     columns=ff_factor_data.columns, 
                     index=excess_returns.columns)
betas

In [0]:
excess_returns.index

In [0]:
for period in betas.index:
    print(period)

In [0]:
for period in excess_returns.index:
    print(period)

In [0]:
excess_returns.index[0]

In [0]:
betas.index

In [0]:
excess_returns.loc['2013-01', betas.index]

In [0]:
betas

In [0]:
lambdas = []
for period in excess_returns.index:
    step2 = OLS(endog=excess_returns.loc[period, betas.index].astype(float), 
                exog=betas).fit()
    lambdas.append(step2.params)

In [0]:
lambdas

In [0]:
lambdas = pd.DataFrame(lambdas, 
                       index=returns.index,
                       columns=betas.columns.tolist())
lambdas

In [0]:
lambdas.mean().sort_values().plot.barh(figsize=(12, 4))
sns.despine()
plt.tight_layout();

In [0]:
t = lambdas.mean().div(lambdas.std())
t

In [0]:
window = 24  # months
ax1 = plt.subplot2grid((1, 3), (0, 0))
ax2 = plt.subplot2grid((1, 3), (0, 1), colspan=2)
lambdas.mean().sort_values().plot.barh(ax=ax1)
lambdas.rolling(window).mean().dropna().plot(lw=1,
                                             figsize=(14, 5),
                                             sharey=True,
                                             ax=ax2)
sns.despine()
plt.tight_layout()

In [0]:
window = 48  # months
ax1 = plt.subplot2grid((1, 3), (0, 0))
ax2 = plt.subplot2grid((1, 3), (0, 1), colspan=2)
lambdas.mean().sort_values().plot.barh(ax=ax1)
lambdas.rolling(window).mean().dropna().plot(lw=1,
                                             figsize=(14, 5),
                                             sharey=True,
                                             ax=ax2)
sns.despine()
plt.tight_layout()

In [0]:
mod = LinearFactorModel(portfolios=excess_returns, 
                        factors=ff_factor_data)
res = mod.fit()
print(res)

In [0]:
print(res.full_summary)
